# N106 · 状态机DP与股票交易

**目标：** 用持有状态和交易次数表达合法历史。

**先修：** N098, N099, N023。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 买卖状态；费用；冷冻期；K次交易；不可达初始化；同日更新顺序。

**配套讲解来源：** [同名逐章意见](../../comment/106_finite_state_dp_stocks.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决的是一族"股票交易"问题：已知一只股票未来每一天的价格（一个数组 `prices`），你从头到尾只能靠"先买后卖"赚钱，问你最多能赚多少。听起来简单，麻烦的是现实里总有一堆附加规则：最多只准交易 K 次；卖完第二天不准买（冷冻期）；每笔交易要交手续费。我们要写四个函数，分别对付这些变体。

先看一个最基础的具体例子，让你对"赚差价"有感觉：

- 输入 `prices = [7,1,5,3,6,4]`（第 0 天 7 块，第 1 天 1 块，以此类推）。
- 输出 `5`。
- 为什么是 5？因为你最优的操作是：第 1 天花 1 块买入，第 4 天 6 块卖出，净赚 6 − 1 = 5。你不能反过来先卖后买，也 不能在第 0 天买入（7 块买入的话后面最高才 6 块，亏）。

再预告一个带冷冻期的例子（第三节会逐天手算它）：

- 输入 `prices = [1,2,3,0,2]`，允许无限次交易，但卖出的第二天必须休息一天才能再买。
- 输出 `3`。最优方案是：第 0 天 1 块买 → 第 1 天 2 块卖（赚 1），第 2 天休息不了？不对，卖完第 1 天，第 2 天是冷冻期不能买，但可以第 2 天什么都不做、第 3 天 0 块买 → 第 4 天 2 块卖（赚 2），合计 3。你想赚 4（1 块买 3 块卖 + 0 块买 2 块卖）是不行的，因为第 2 天卖出后第 3 天是冷冻期，买不了 0 块那一次。

## 2. 基础知识：先读懂这些词

- **状态机 DP（状态机动态规划）**：把"你现在处于什么处境"抽象成几个状态（比如"手里有股票"和"手里没股票"），每天从一个状态跳到另一个状态。整个交易过程就是状态机上走的一条路，我们要求"走完 n 天后钱最多"的那条路。
- **状态（持仓 / 现金）**：本章最核心的两个状态是"持仓"（手里攥着股票，钱已经付出去了，这个变量的值是"总收益"，所以买入时要减去价格）和"现金/未持仓"（手里只有钱）。你必须知道自己在哪个状态，才能知道今天允许做什么动作。
- **交易次数维度**：在"最多 K 次交易"的版本里，光知道持仓与否不够，还得知道"已经用掉几次交易了"，否则没法限制次数。所以那版的状态是二维的：`buy[t]`、`sell[t]` 里的 `t` 就是已用的交易次数。
- **不可达初始化（用 `float('-inf')`）**：负无穷在这里的意思是"这个状态根本到不了，别考虑它"。比如第 0 天开始前你不可能已经持有股票，所以初始持仓收益是 `-inf`，任何正常方案和它取 `max` 时都会赢过它，等于自动把非法路径排除掉。
- **同日更新顺序（元组赋值读旧值）**：`hold, sold, rest = f(hold,...), g(...), h(...)` 这种写法里，右边的所有值都是"今天还没更新之前"的旧值。这保证了今天的转移只读昨天的状态，你就不可能在同一天"刚买又立刻卖"刷出非法利润。
- **冷冻期（cooldown）**：卖出股票后的下一个交易日禁止买入，必须至少休息一天。为了表示"今天是刚卖出的、明天不能买"，需要单独加一个"刚售出"状态。
- **手续费（fee）**：每完成一次"买+卖"，交易所扣一次固定费用。注意只扣一次，不能买入扣一次卖出又扣一次。
- **K 足够大时退化为无限交易**：一笔交易至少要占两天（一天买、一天卖），所以 n 天里最多做 `n//2` 笔交易。当题目给的 K ≥ n//2 时，"K 次限制"等于没限制，直接用"见涨就赚"的贪心即可，这就是代码里那句 `if k>=len(prices)//2` 的含义。

## 3. 思路推导：从小例子开始

我们以冷冻期版本为例，一步步把状态机推出来。

- **Step 1：定状态。** 每天结束时你只可能处于三种处境之一：`rest`（空仓且可买，啥事没有）、`hold`（手里有股票）、`sold`（今天刚卖出，明天进入冷冻期）。三个变量分别存"处于该状态时的最大累计收益"。
- **Step 2：定初始值。** 第一天开始前，你必然空仓且收益为 0，所以 `rest=0`；`hold` 和 `sold` 都是 `-inf`，表示"还没开始就持仓/刚卖出"是非法的，别考虑。
- **Step 3：定转移。** 每天有三种动作——不干活、买入、卖出，对每个新状态取所有能到达它的动作里的最优：
  - 新 `hold` = max(昨天的 `hold`（继续拿）， 昨天的 `rest` − 今天价格（今天买入）)。注意不能从 `sold` 买，因为刚卖出要冷冻一天。
  - 新 `sold` = 昨天的 `hold` + 今天价格（只有持仓者今天能卖）。
  - 新 `rest` = max(昨天的 `rest`， 昨天的 `sold`（冷冻一天后恢复自由）)。
- **Step 4：手算一遍。** 下面这张表就是 “运行示例”部分代码对 `prices=[1,2,3,0,2]` 生成的真实轨迹，我们一格一格看（`-inf` 表示该状态当天不可达）：

| 天 | 价格 | 持仓 | 刚售出 | 可买入 |
|---|---|---|---|---|
| 0 | 1 | -1 | -inf | 0 |
| 1 | 2 | -1 | 1 | 0 |
| 2 | 3 | -1 | 2 | 1 |
| 3 | 0 | 1 | -1 | 2 |
| 4 | 2 | 1 | 3 | 2 |

逐行验证：第 0 天价格 1，持仓 = max(-inf, 0−1) = −1（花 1 块买了）；刚售出 = 旧持仓+1 = -inf（之前没持仓，卖不了）；可买入 = 0。第 1 天价格 2：持仓 = max(−1, 0−2) = −1（继续拿更划算）；刚售出 = −1+2 = 1（昨天 1 块买的今天 2 块卖，赚 1）；可买入 = 0。第 2 天价格 3：持仓 = max(−1, 0−3) = −1；刚售出 = −1+3 = 2；可买入 = max(0, 1) = 1（昨天的"刚售出"今天解冻）。第 3 天价格 0：持仓 = max(−1, 1−0) = 1（解冻的钱抄底 0 块的股票！）；刚售出 = −1+0 = −1；可买入 = max(1, 2) = 2。第 4 天价格 2：持仓 = max(1, 2−2) = 1；刚售出 = 1+2 = 3；可买入 = max(2, −1) = 2。

- **Step 5：读答案。** 最后一天结束时手里必须没股票（拿着股票不卖的钱是纸面浮盈），所以答案 = max(0, sold, rest) = max(0, 3, 2) = 3，和我们第一节里手推的最优方案一致。

## 4. 核心代码：max_profit_once

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def max_profit_once(prices):
    low = float('inf')
    best = 0
    for p in prices:
        low = min(low, p)
        best = max(best, p - low)
    return best
```

### 逐段读懂代码

### 1. 只交易一次：`max_profit_once`

```python
def max_profit_once(prices):
    low = float('inf')
    best = 0
    for p in prices:
        low = min(low, p)
        best = max(best, p - low)
    return best
```

这个函数一边扫一边维护两个量：`low` 是"到目前为止见过的最低价"，`best` 是"如果今天卖，历史最好利润"。每一天先用 `min(low,p)` 把最低价刷新，再算"今天卖能赚多少"（`p-low`，即今天价格减历史最低价），用 `max` 记录最大值。`best=0` 的意思是"最差我一次都不交易，收益为 0"，它同时挡住了价格一直跌的情况（此时任何交易都亏，不如不买）。注意 `low` 初始化为正无穷 `float('inf')`，保证第一天的价格一定能成为初始最低价。

### 2. 最多 K 次交易：`max_profit_k`

```python
def max_profit_k(prices, k):
    if k <= 0:
        return 0
    if k >= len(prices) // 2:
        return sum((max(0, b - a) for a, b in zip(prices, prices[1:])))
    buy = [float('-inf')] * (k + 1)
    sell = [0] * (k + 1)
    for p in prices:
        old_buy, old_sell = (buy, sell)
        buy = old_buy[:]
        sell = old_sell[:]
        for t in range(1, k + 1):
            buy[t] = max(old_buy[t], old_sell[t - 1] - p)
            sell[t] = max(old_sell[t], old_buy[t] + p)
    return sell[k]
```

逐段看：

- `if k<=0: return 0`：一次交易都不许做，那收益只能是 0，这是挡非法输入的边界判断。
- `if k>=len(prices)//2: ...`：上面说过，n 天最多做 n//2 笔交易，所以 K 给得比这还大就等于无限次。此时最优策略是贪心：把每对相邻两天 `zip(prices,prices[1:])` 里"明天比今天贵"的差价 `b-a` 全部收下（负差价跳过，`max(0,...)`），加起来就是答案。
- `buy=[float('-inf')]*(k+1)` 和 `sell=[0]*(k+1)`：按下标 `t`（已完成的交易数）分层。`buy[t]` 是"正在做第 t 笔交易、手里有票"的最大收益，初始负无穷表示不可达；`sell[t]` 是"已完成 t 笔、空仓"的最大收益，`sell[0]=0` 表示一笔没做、收益 0。
- 循环体里先 `old_buy,old_sell=buy,sell; buy=old_buy[:]; sell=old_sell[:]` 把昨天的数组复制一份，接下来更新时全部读 `old_` 版本。这和冷冻版元组赋值是同一个目的：今天只准基于昨天转移，防止同一天"买了马上卖"白嫖。
- `buy[t]=max(old_buy[t],old_sell[t-1]-p)`：今天结束仍持仓且已启动第 t 笔，要么昨天就持仓（`old_buy[t]`），要么昨天空仓且已完成 t−1 笔、今天买入（`old_sell[t-1]-p`）。从 `t-1` 层拿钱，正是"这笔交易计数 +1"的体现。
- `sell[t]=max(old_sell[t],old_buy[t]+p)`：完成第 t 笔有两种来路——昨天就完成了（不动），或昨天持有第 t 笔的股票、今天卖出（`old_buy[t]+p`）。
- `return sell[k]`：注意返回的是 `sell[k]` 而不是 `max(sell)`，因为做满 k 笔未必最优——不过 `sell[t]` 的转移里"不动"这一支允许它一直把少交易的优势传到 `sell[k]`，所以少交易的最优值也会流到这里。

顺手用 `prices=[3,2,6,5,0,3]`、k=2 手算一遍（n=6，k < n//2=3，走 DP 分支）：

- 价格走到 6 那天，`sell[1]=4`（第一笔的最佳：2 块买 6 块卖）。
- 价格 0 那天，`buy[2]=4`（"正在做第二笔、手里有票"的最佳收益：拿第一笔赚的 4 块再花 0 块买一股，账面 4）。
- 价格 3 那天收尾，`sell[2]=4+3=7`（0 块买的股票 3 块卖出）。答案 7，对应两笔交易"2→6"加"0→3"。

### 3. 带冷冻期：`max_profit_cooldown`

```python
def max_profit_cooldown(prices):
    hold = sold = float('-inf')
    rest = 0
    for p in prices:
        hold, sold, rest = (max(hold, rest - p), hold + p, max(rest, sold))
    return max(0, sold, rest)
```

这就是第三节手算的那台状态机。三个变量的含义见第三节。关键在那一行三连赋值：Python 会先把右边三个表达式全部算完（用的全是旧值）再统一赋给左边，所以 `sold` 拿到的 `hold+p` 是"昨天的持仓"，天然满足"当天转移读前一天状态"的要求。最后 `max(0,sold,rest)` 表示空仓结束，`0` 这一项兜底"全程不交易"，`hold` 不参与是因为持仓未卖的钱不算落袋。

### 4. 带手续费：`max_profit_fee`

```python
def max_profit_fee(prices, fee):
    hold = float('-inf')
    cash = 0
    for p in prices:
        hold, cash = (max(hold, cash - p), max(cash, hold + p - fee))
    return cash
```

只有两个状态：`hold` 持仓、`cash` 空仓现金。买入是 `cash-p`（现金减少），卖出是 `hold+p-fee`（股票变现并扣一次手续费）。注意 `fee` 只出现在卖出那一支转移里，这就是"费用只扣一次"的代码表达。同样靠元组赋值保证同日不连环操作。返回 `cash`，理由同上：只有空仓的收益是真正到手的。

## 5. 分段实现：按顺序运行

**本章接口：** `max_profit_once(prices)`、`max_profit_k(prices, k)`、`max_profit_cooldown(prices)`、`max_profit_fee(prices, fee)`

### max_profit_once

In [1]:
def max_profit_once(prices):
    low = float('inf')
    best = 0
    for p in prices:
        low = min(low, p)
        best = max(best, p - low)
    return best

### max_profit_k

In [2]:
def max_profit_k(prices, k):
    if k <= 0:
        return 0
    if k >= len(prices) // 2:
        return sum((max(0, b - a) for a, b in zip(prices, prices[1:])))
    buy = [float('-inf')] * (k + 1)
    sell = [0] * (k + 1)
    for p in prices:
        old_buy, old_sell = (buy, sell)
        buy = old_buy[:]
        sell = old_sell[:]
        for t in range(1, k + 1):
            buy[t] = max(old_buy[t], old_sell[t - 1] - p)
            sell[t] = max(old_sell[t], old_buy[t] + p)
    return sell[k]

### max_profit_cooldown

In [3]:
def max_profit_cooldown(prices):
    hold = sold = float('-inf')
    rest = 0
    for p in prices:
        hold, sold, rest = (max(hold, rest - p), hold + p, max(rest, sold))
    return max(0, sold, rest)

### max_profit_fee

In [4]:
def max_profit_fee(prices, fee):
    hold = float('-inf')
    cash = 0
    for p in prices:
        hold, cash = (max(hold, cash - p), max(cash, hold + p - fee))
    return cash

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

prices=[1,2,3,0,2]; hold=sold=float('-inf'); rest=0; rows=[]
for day,p in enumerate(prices):
    hold,sold,rest=max(hold,rest-p),hold+p,max(rest,sold); rows.append((day,p,hold,sold,rest))
show_table(['天','价格','持仓','刚售出','可买入'],rows)

天,价格,持仓,刚售出,可买入
0,1,-1,-inf,0
1,2,-1,1,0
2,3,-1,2,1
3,0,1,-1,2
4,2,1,3,2


## 7. 正确性、适用条件与复杂度

每个合法交易过程映射到状态机的一条路径。枚举休息、买入、卖出三种动作完整覆盖合法路径，非法转移被状态限制排除。冷冻状态只能在卖出后经过一天休息再买；元组赋值右侧都读取旧值。

**代价：** 单次、冷冻和费用版O(n)时间O(1)辅助；k次版O(nk)时间O(k)辅助，k足够大时退化为无限交易O(n)。价格非负，手续费非负，交易必须先买后卖。

### 展开理解

**为什么是对的？** 我们论证的思路是：任何一种合法的交易方案，都可以看成这台状态机上真实走过的一条路线，路线的终点收益就是方案的利润；反过来，状态机上每一条合法路线也对应一个合法方案。所以"所有路线里的最大收益"和"所有方案里的最大利润"是同一个数。接下来只要确认两件事：第一，每天你真正能做的动作只有"不动、买、卖"三种，转移方程把这三种全列出来了，没有漏掉任何选择；第二，做不到的动作（比如空仓想卖、冷冻期想买、次数用完还想买）对应的转移分支根本不存在，非法方案压根进不了我们的候选池。初始值用负无穷把"开局就持仓"这类不可能的起点标记成永远选不中的值。至于"当天读昨天的值"，元组赋值和复制旧数组的写法从机制上保证了这一点，我们就不需要额外担心同日刷单。

**前提条件**：代码假设价格非负、手续费非负、交易必须先买后卖。这些是题目给的约束，不是算法自带的。

**复杂度**：`max_profit_once`、`max_profit_cooldown`、`max_profit_fee` 每天只做常数次比较，所以是 O(n) 时间、O(1) 额外空间。`max_profit_k` 每天要更新 k 层，是 O(n·k) 时间、O(k) 空间。拿具体数字感受一下：n = 10⁵（十万天的价格）、k = 100 时，`max_profit_k` 大约要做一千万次基本操作，现代计算机一眨眼就算完；而三个 O(n) 版本只要十万次操作，快到可以忽略。另外当 k ≥ n//2 时走贪心分支，退回 O(n)。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分的测试分两部分。第一部分是三条具体断言：

- `assert max_profit_once([7,1,5,3,6,4])==5`：这是正常值测试，就是第一节我们手算的例子——最低点 1 买、最高点 6 卖，利润 5。它检查"单次交易取全局最优"这条主线逻辑。
- `assert max_profit_cooldown([1,2,3,0,2])==3`：这是特殊规则测试，对应第三节那张手算表。如果你忘了冷冻期限制（比如允许第 2 天卖完第 3 天立刻买），会错误地算出 4，这条就能抓住你。
- `assert max_profit_fee([1,3,2,8,4,9],2)==8`：这是手续费测试。最优是两笔：1 买 8 卖赚 8−1−2=5，4 买 9 卖赚 9−4−2=3，合计 8。如果手续费被错误地扣两次（买卖各一次），答案会变小，这条就能抓住。

第二部分是与暴力解 `brute` 的全量对拍。`brute` 的递归函数 `f(i,holding,remaining,blocked)` 有四个参数：`i` 是当前天数；`holding` 是是否持仓；`remaining` 是剩余交易次数；`blocked` 是"今天是否被冷冻"（只有在开启冷冻模式的对拍里才可能为 True）。它每天只做三件事：什么都不做推进到下一天；持仓且有剩余次数时卖出（收 `prices[i]-fee`，若冷冻模式则下一天 blocked）；空仓、没被冷冻且有剩余次数时买入。基例是走完最后一天：还持仓说明方案非法，返回负无穷，否则返回 0。这个参照实现与本章四个函数是**独立**写的（它不懂什么状态机），所以对拍才有说服力。

测试用 `product(range(3),repeat=5)` 枚举了所有长度为 5、价格取 0/1/2 的 3⁵=243 种价格数组，对每一种都做四组对比：`max_profit_once` 对 `brute(prices,1)`（只准一笔）；`max_profit_k(prices,2)` 对 `brute(prices,2)`（恰好两笔上限）；`max_profit_fee(prices,1)` 对 `brute(prices,fee=1)`（无限笔、每笔 1 元手续费）；`max_profit_cooldown(prices)` 对 `brute(prices,cooldown=True)`（无限笔、带冷冻）。这种小规模全枚举对拍是验证 DP 正确性最省心的办法：小输入上穷举不会错，你的实现和它一致，基本就稳了。

In [6]:
assert max_profit_once([7, 1, 5, 3, 6, 4]) == 5

assert max_profit_cooldown([1, 2, 3, 0, 2]) == 3

assert max_profit_fee([1, 3, 2, 8, 4, 9], 2) == 8

from itertools import product

from functools import cache

def brute(prices, k=99, fee=0, cooldown=False):

    @cache
    def f(i, holding, remaining, blocked):
        if i == len(prices):
            return float('-inf') if holding else 0
        best = f(i + 1, holding, remaining, False)
        if holding and remaining:
            best = max(best, prices[i] - fee + f(i + 1, False, remaining - 1, cooldown))
        if not holding and (not blocked) and remaining:
            best = max(best, -prices[i] + f(i + 1, True, remaining, False))
        return best
    return f(0, False, k, False)

for prices in product(range(3), repeat=5):
    assert max_profit_once(prices) == brute(prices, 1)
    assert max_profit_k(prices, 2) == brute(prices, 2)
    assert max_profit_fee(prices, 1) == brute(prices, fee=1)
    assert max_profit_cooldown(prices) == brute(prices, cooldown=True)

print('N106: 所有本章断言通过')

N106: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 从统一状态机派生无限次交易。

**练习 2：** 解释初始持有不能初始化为0。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（从统一状态机派生无限次交易）**：提示——回头看 `max_profit_fee`，把 `fee` 设成 0 会发生什么？那两个转移方程就是无限次交易的状态机。你要做的是：明确输入输出（价格数组进、最大利润出）、给一个手算例子（比如 `[1,2,3,0,2]` 不带冷冻期时答案应为 4，想想为什么比带冷冻期的 3 多）、再说明边界（价格单调递减时应返回 0）。验证时可以直接调 `max_profit_fee(prices, 0)` 或 `max_profit_k(prices, 很大的k)` 当参照。
- **练习 2（解释初始持有为什么不能初始化为 0）**：提示——假设把 `hold` 的初始值从 `-inf` 改成 0，手算 `[5,1]` 这种输入：第一天"持仓收益 0"意味着你凭空拥有一张免费股票，第二天以 5 卖出会得到利润 5，但真实世界里你得先花 5 块买。也就是说，初始化成 0 相当于允许"没花钱就拿到股票"，答案会系统性偏大。你要把这个"凭空白送"的失败例子写出来，并说明 `float('-inf')` 如何通过"max 永远选不中它"来堵住这条非法路径。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
def max_profit_once(prices):
    low = float('inf')
    best = 0
    for p in prices:
        low = min(low, p)
        best = max(best, p - low)
    return best

def max_profit_k(prices, k):
    if k <= 0:
        return 0
    if k >= len(prices) // 2:
        return sum((max(0, b - a) for a, b in zip(prices, prices[1:])))
    buy = [float('-inf')] * (k + 1)
    sell = [0] * (k + 1)
    for p in prices:
        old_buy, old_sell = (buy, sell)
        buy = old_buy[:]
        sell = old_sell[:]
        for t in range(1, k + 1):
            buy[t] = max(old_buy[t], old_sell[t - 1] - p)
            sell[t] = max(old_sell[t], old_buy[t] + p)
    return sell[k]

def max_profit_cooldown(prices):
    hold = sold = float('-inf')
    rest = 0
    for p in prices:
        hold, sold, rest = (max(hold, rest - p), hold + p, max(rest, sold))
    return max(0, sold, rest)

def max_profit_fee(prices, fee):
    hold = float('-inf')
    cash = 0
    for p in prices:
        hold, cash = (max(hold, cash - p), max(cash, hold + p - fee))
    return cash

# 示例与回归测试
assert max_profit_once([7, 1, 5, 3, 6, 4]) == 5

assert max_profit_cooldown([1, 2, 3, 0, 2]) == 3

assert max_profit_fee([1, 3, 2, 8, 4, 9], 2) == 8

from itertools import product

from functools import cache

def brute(prices, k=99, fee=0, cooldown=False):

    @cache
    def f(i, holding, remaining, blocked):
        if i == len(prices):
            return float('-inf') if holding else 0
        best = f(i + 1, holding, remaining, False)
        if holding and remaining:
            best = max(best, prices[i] - fee + f(i + 1, False, remaining - 1, cooldown))
        if not holding and (not blocked) and remaining:
            best = max(best, -prices[i] + f(i + 1, True, remaining, False))
        return best
    return f(0, False, k, False)

for prices in product(range(3), repeat=5):
    assert max_profit_once(prices) == brute(prices, 1)
    assert max_profit_k(prices, 2) == brute(prices, 2)
    assert max_profit_fee(prices, 1) == brute(prices, fee=1)
    assert max_profit_cooldown(prices) == brute(prices, cooldown=True)

print('N106: 所有本章断言通过')

N106: 所有本章断言通过


## 11. 代表题与迁移

- **121. Best Time to Buy and Sell Stock**：只准买卖一次，练的是 `max_profit_once` 里"扫一遍维护历史最低价"的套路。
- **122. Best Time to Buy and Sell Stock II**：无限次交易，练的是"K 足够大时退化贪心"那条分支（相邻正差价求和），也可以用 `fee=0` 的两状态机。
- **188. Best Time to Buy and Sell Stock IV**：最多 k 次交易，练的是 `buy[t]/sell[t]` 的分层状态与"当天用旧数组"的更新纪律。
- **309. Best Time to Buy and Sell Stock with Cooldown**：冷冻期，练的是三状态机（持仓/刚售出/可买入）以及元组赋值同日读旧值的技巧。
- **714. Best Time to Buy and Sell Stock with Transaction Fee**：手续费，练的是在卖出转移里一次性扣 `fee`，其余与两状态机完全同构。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。